# CS 553 - Neural Networks Project 1

This notebooks contains code that aims to mimic the results of the MedMNIST V2 paper. We will reproduce and train a ResNet-18 model on the VessleMNIST dataset utilizing 3D convolutions. The module we aim to use is pytorch.

## Image Preprocessing

In [ ]:
#import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
from medmnist import VesselMNIST3D
from torch.utils.data import Dataset
from acsconv.converters import ACSConverter, Conv3dConverter
import torchvision.models as models 

In [ ]:
#Import VesselMNIST training dataset 
train_dataset = VesselMNIST3D(split="train", download=True)
train_dataset

In [ ]:
#import VessleMNIST val dataset
val_dataset = VesselMNIST3D(split="val", download= True)
val_dataset

In [ ]:
#import VessleMNIST testing dataset
test_dataset = VesselMNIST3D(split="test", download= True)
test_dataset

In [ ]:
#Look at the first few samples of the dataset
for i in range(5):
    sample_image, sample_label = train_dataset[i]
    print(f'sample image', sample_image, 'sample_label', sample_label)

In [ ]:
sample_image, sample_label = train_dataset[0]
sample_image

In [ ]:
#Create class that expands the image channel from 1 -> 3
#Data augmentation and normalization
class ExpandedVesselMNIST(Dataset):
    def __init__(self, base_dataset, is_training = True):
        self.base_dataset = base_dataset
        self.is_training = is_training
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        tensor_image = torch.from_numpy(image).float()

        if self.is_training:
            rand_factor = torch.rand(1).item()
            tensor_image = tensor_image * rand_factor
        else:
            tensor_image = tensor_image * 0.5

        expanded_image = tensor_image.repeat(3, 1, 1, 1)
        return expanded_image, torch.tensor(label).long()

In [ ]:
#Expand image channels for training dataset 
mod_train_set = ExpandedVesselMNIST(train_dataset, is_training=True)
mod_val_set = ExpandedVesselMNIST(val_dataset, is_training=False)
mod_test_set = ExpandedVesselMNIST(test_dataset, is_training=False)

# Model Architecture and Training: ResNet-18 with 3D Convolutions

In [ ]:
#Import resnet18 utilizing an untrained neural network 
resnet18 = models.resnet18(pretrained=False)
#Convert model into 3D convolutional neural network
model_3d = Conv3dConverter(resnet18)